In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error


# ==================================================
# 2. 원본 데이터 다시 불러오기
# ==================================================

train_raw = pd.read_csv('modulabs-di/train.csv')
test_raw = pd.read_csv('modulabs-di/test.csv')

print("Train 데이터:", train_raw.shape)
print("Test 데이터:", test_raw.shape)

Train 데이터: (1168, 81)
Test 데이터: (292, 80)


In [7]:
# ==================================================
# 3. 정답 데이터와 예측 데이터 분리
# ==================================================

# 정답 데이터
y = train_raw['SalePrice']

# 학습 데이터에서 SalePrice와 Id 제거
X = train_raw.drop(columns=['SalePrice', 'Id'])

# 테스트 데이터의 Id는 제출 파일에 사용하므로 별도 저장
test_ids = test_raw['Id']

# 테스트 데이터의 예측 변수
X_test = test_raw.drop(columns=['Id'])

# 학습 데이터와 테스트 데이터의 컬럼 구조 맞추기
X_test = X_test.reindex(columns=X.columns)


In [8]:
# ==================================================
# 4. 컬럼 유형 구분
# ==================================================

# 범주형 컬럼과 수치형 컬럼 구분
categorical_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numeric_cols = X.select_dtypes(
    include='number'
).columns.tolist()


# 시설이 없는 경우를 의미하는 범주형 컬럼
none_cat_candidates = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual',
    'GarageCond', 'PoolQC', 'Fence', 'MiscFeature',
    'MasVnrType'
]

# 시설이 없는 경우 0으로 처리할 수치형 컬럼
zero_num_candidates = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
    'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath',
    'GarageYrBlt', 'GarageCars', 'GarageArea',
    'MasVnrArea'
]

# 실제 데이터에 존재하는 컬럼만 선택
none_cat_cols = [
    col for col in none_cat_candidates
    if col in categorical_cols
]

zero_num_cols = [
    col for col in zero_num_candidates
    if col in numeric_cols
]

# 나머지 범주형과 수치형 컬럼
mode_cat_cols = [
    col for col in categorical_cols
    if col not in none_cat_cols
]

median_num_cols = [
    col for col in numeric_cols
    if col not in zero_num_cols
]


C:\Users\chaee\AppData\Local\Temp\ipykernel_5024\2389888998.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [9]:
# ==================================================
# 5. 전처리 과정 설정
# ==================================================

# 시설이 없는 범주형 컬럼: None 처리 + 원 핫 인코딩
cat_none_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value='None'
    )),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

# 일반 범주형 컬럼: 최빈값 처리 + 원 핫 인코딩
cat_mode_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='most_frequent'
    )),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

# 시설이 없는 수치형 컬럼: 0 처리 + 정규화
num_zero_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='constant',
        fill_value=0
    )),
    ('scaler', StandardScaler())
])

# 일반 수치형 컬럼: 중앙값 처리 + 정규화
num_median_pipeline = Pipeline([
    ('imputer', SimpleImputer(
        strategy='median'
    )),
    ('scaler', StandardScaler())
])


# 모든 전처리 과정 통합
preprocessor = ColumnTransformer([
    ('cat_none', cat_none_pipeline, none_cat_cols),
    ('cat_mode', cat_mode_pipeline, mode_cat_cols),
    ('num_zero', num_zero_pipeline, zero_num_cols),
    ('num_median', num_median_pipeline, median_num_cols)
])



In [39]:
# ==================================================
# 6. 모델 설정
# ==================================================

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=50))
])



In [40]:
# ==================================================
# 7. 학습 데이터와 검증 데이터 분리
# ==================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



In [41]:
# ==================================================
# 8. 모델 학습
# ==================================================

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](79,)","['MSSubClass','MSZoning','LotFrontage',...,'YrSold','SaleType', 'SaleCondition']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,79
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_none', ...), ('cat_mode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed thr

In [42]:
# ==================================================
# 9. 검증 데이터 예측 및 RMSE 평가
# ==================================================

valid_pred = model.predict(X_valid)

rmse = np.sqrt(
    mean_squared_error(y_valid, valid_pred)
)

print(f"Validation RMSE: {rmse:.2f}")


Validation RMSE: 22117.76


In [43]:
# ==================================================
# 10. 전체 학습 데이터로 최종 모델 학습
# ==================================================

model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](79,)","['MSSubClass','MSZoning','LotFrontage',...,'YrSold','SaleType', 'SaleCondition']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,79
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_none', ...), ('cat_mode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed thr

In [44]:
# ==================================================
# 11. 테스트 데이터 예측
# ==================================================

test_pred = model.predict(X_test)

In [45]:
# ==================================================
# 12. 제출 파일 생성
# ==================================================

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': test_pred
})

submission.to_csv(
    'submission.csv',
    index=False
)

print("\n제출 파일 생성 완료!")
print(submission.head())
print("\n제출 파일 크기:", submission.shape)


제출 파일 생성 완료!
    Id      SalePrice
0   86  256797.313838
1  227  277831.472929
2  492  156941.440599
3   64  128712.074668
4  576  108110.777867

제출 파일 크기: (292, 2)
